# 第9章　可赎回债券与可回售债券

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch09_callable_putable.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch09_callable_putable.ipynb)

复现例10.1（可赎回）、例10.2（可回售）、图9-1（负凸性）、有效凸性扫描，以及 QuantLib OAS 对波动率的敏感性。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi import tree, plotting
plotting.use_chinese_style()


## 例10.1　可赎回债定价与久期压缩

6 年期、年付息 6%，r0=3%、σ=20%（低利率高票息），赎回价 100、第 1 步起可赎回。


In [ ]:
t = tree.short_rate_tree(r0=0.03, sigma=0.20, n_steps=6)
straight = tree.value_bond(t, 6.0, 100)
callable_ = tree.value_bond(t, 6.0, 100, call_price=100, call_from=1)
print(f'普通债   = {straight:.4f}（溢价）')
print(f'可赎回债 = {callable_:.4f}')
print(f'赎回期权价值 = {straight - callable_:.4f}')
ed_s = tree.effective_duration_tree(0.03, 0.20, 6, 6.0)
ed_c = tree.effective_duration_tree(0.03, 0.20, 6, 6.0, call_price=100, call_from=1)
print(f'有效久期: 普通债={ed_s:.4f}  可赎回={ed_c:.4f}（久期被压缩）')


## 例10.2　可回售债定价与价格托底

6 年期、年付息 2%，r0=5%（高利率低票息，普通债深度折价），回售价 100、第 1 步起可回售。


In [ ]:
t2 = tree.short_rate_tree(0.05, 0.20, 6)
s2 = tree.value_bond(t2, 2.0, 100)
p2 = tree.value_bond(t2, 2.0, 100, put_price=100, put_from=1)
print(f'普通债   = {s2:.4f}（折价）')
print(f'可回售债 = {p2:.4f}')
print(f'回售期权价值 = {p2 - s2:.4f}')
ed_s2 = tree.effective_duration_tree(0.05, 0.20, 6, 2.0)
ed_p2 = tree.effective_duration_tree(0.05, 0.20, 6, 2.0, put_price=100, put_from=1)
print(f'有效久期: 普通债={ed_s2:.4f}  可回售={ed_p2:.4f}')


## 图9-1　负凸性：可赎回 vs 普通的价格—利率曲线（编程实验 7）


In [ ]:
refs = np.linspace(0.01, 0.08, 36)
sp = [tree.value_bond(tree.short_rate_tree(r, 0.20, 6), 6.0, 100) for r in refs]
cp = [tree.value_bond(tree.short_rate_tree(r, 0.20, 6), 6.0, 100, call_price=100, call_from=1) for r in refs]
fig, ax = plotting.new_axes()
ax.plot(refs*100, sp, label='普通债')
ax.plot(refs*100, cp, label='可赎回债（赎回价 100）')
ax.axhline(100, ls=':', color='gray')
ax.set_xlabel('短期利率 r0 (%)'); ax.set_ylabel('价格')
ax.set_title('图9-1　可赎回债的负凸性'); ax.legend()
fig.tight_layout()


### 有效凸性扫描：何处由正转负（编程实验 8）


In [ ]:
print('可赎回债有效凸性随 r0:')
for rr in [0.02, 0.03, 0.04, 0.05, 0.06, 0.07]:
    ec = tree.effective_convexity_tree(rr, 0.20, 6, 6.0, dy=2e-3, call_price=100, call_from=1)
    print(f'  r0={rr:.0%}: 有效凸性={ec:9.2f}' + ('  <-- 负凸性（期权价内附近）' if ec < 0 else ''))


## 10.7　QuantLib：CallableFixedRateBond 与 OAS 对波动率的敏感性（编程实验 9）


In [ ]:
import QuantLib as ql
today = ql.Date(15, 6, 2026); ql.Settings.instance().evaluationDate = today
dc = ql.ActualActual(ql.ActualActual.ISDA)
ts = ql.YieldTermStructureHandle(ql.FlatForward(today, 0.03, dc))
sched = ql.Schedule(today, today + ql.Period(6, ql.Years), ql.Period(ql.Annual),
                    ql.NullCalendar(), ql.Unadjusted, ql.Unadjusted, ql.DateGeneration.Backward, False)
calls = ql.CallabilitySchedule()
for y in range(1, 6):
    calls.append(ql.Callability(ql.BondPrice(100.0, ql.BondPrice.Clean),
                                ql.Callability.Call, today + ql.Period(y, ql.Years)))
bond = ql.CallableFixedRateBond(0, 100.0, sched, [0.06], dc, ql.Unadjusted, 100.0, today, calls)
bond.setPricingEngine(ql.TreeCallableFixedRateBondEngine(ql.HullWhite(ts, 0.03, 0.015), 100))
mkt = bond.cleanPrice()
print(f'QuantLib 可赎回债 clean price = {mkt:.4f}')
print('\n固定市价，OAS 随波动率假设变化:')
for vol in (0.005, 0.015, 0.03):
    bond.setPricingEngine(ql.TreeCallableFixedRateBondEngine(ql.HullWhite(ts, 0.03, vol), 100))
    oas = bond.OAS(mkt, ts, dc, ql.Compounded, ql.Annual, today, 1e-10, 200, 0.0) * 1e4
    print(f'  vol={vol*100:.1f}%: OAS = {oas:.2f} bp')


---

> 小结：含权债用利率树反向归纳定价，可赎回取 min(续作,赎回价)、可回售取 max(续作,回售价)；
> 可赎回债低利率端负凸性、久期压缩；OAS 剥离期权价值但依赖波动率假设。
